# Secure dataset handling: classification, minimisation, controlled loading

**Module 05 — Secure Data Management and Dataset Protection**

Most real leaks do not come from a clever attack on a model. They come from a table that was loaded with
more columns than the job needed, written to a shared drive, and joined to something else six months
later. This notebook builds the boring controls that prevent it: a column classification, a minimisation
policy enforced in code, and a loader that refuses to hand out columns it was not asked for.

## Learning objectives

1. Classify every column of a dataset by sensitivity and justify the classification.
2. Enforce column minimisation in a single loader rather than at every call site.
3. Detect and refuse policy violations (over-broad read, unsupported type, disallowed value).
4. Produce a dataset card that a reviewer can challenge.
5. Explain why de-identification belongs in the pipeline, not in a downstream notebook.

## Background

Module 05 separates three ideas that are often merged:

* **Classification** — what kind of data is this, and who may see it?
* **Minimisation** — the smallest set of columns that satisfies the purpose.
* **Enforcement** — the rule is applied by code that fails closed, not by convention.

The dataset here is synthetic and deliberately messy: it contains direct identifiers (a name), quasi
identifiers (postcode, age), a sensitive attribute (diagnosis) and an outcome label.

## Dependencies and installation

```bash
python -m pip install "pandas>=2.2" "numpy>=2.1"
```

CPU, sub-second runtime, no network, no external data.

In [1]:
import hashlib
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 5
rng = np.random.default_rng(SEED)

workdir = Path(tempfile.mkdtemp(prefix="sas-m05a-"))
print("workspace:", workdir.name)

workspace: sas-m05a-j4hwd3jm


**Expected output** — a temporary directory name. Everything written by this notebook stays inside it.

### Step 1 — Build a synthetic dataset that looks like the real problem

Eight rows is enough to reason about by hand and too small to hide a mistake.

In [2]:
data = pd.DataFrame(
    {
        "patient_id": [f"P{100+i}" for i in range(8)],
        "full_name": ["Ayesha Khan", "Bilal Ahmed", "Chen Wei", "Diego Ruiz",
                      "Elena Petrova", "Farah Siddiqui", "Gita Nair", "Hassan Ali"],
        "age": [34, 51, 62, 45, 29, 71, 58, 40],
        "postcode": ["54000", "54000", "74200", "54000", "74200", "54000", "74200", "74200"],
        "diagnosis": ["asthma", "diabetes", "cardiac", "asthma", "cardiac",
                      "diabetes", "asthma", "cardiac"],
        "readmitted": [0, 1, 1, 0, 1, 1, 0, 1],
    }
)
data

,patient_id,full_name,age,postcode,diagnosis,readmitted
0,P100,Ayesha Khan,34,54000,asthma,0
1,P101,Bilal Ahmed,51,54000,diabetes,1
2,P102,Chen Wei,62,74200,cardiac,1
3,P103,Diego Ruiz,45,54000,asthma,0
4,P104,Elena Petrova,29,74200,cardiac,1
5,P105,Farah Siddiqui,71,54000,diabetes,1
6,P106,Gita Nair,58,74200,asthma,0
7,P107,Hassan Ali,40,74200,cardiac,1


**Expected output** — eight rows. `full_name` is a direct identifier, `postcode` and `age` are quasi
identifiers that combine to identify, `diagnosis` is sensitive, and `readmitted` is the label a study
might legitimately need.

### Step 2 — Classify columns as data, not comments

The classification lives in a table so that a reviewer can disagree with a specific row. Note `reason`:
a classification without a reason is not auditable.

In [3]:
classification = pd.DataFrame(
    [
        ("patient_id", "identifier", "unique key, joins to the EHR"),
        ("full_name", "direct_identifier", "names a person"),
        ("age", "quasi_identifier", "narrows a population when combined"),
        ("postcode", "quasi_identifier", "narrows a population when combined"),
        ("diagnosis", "sensitive", "health status"),
        ("readmitted", "label", "the outcome under study"),
    ],
    columns=["column", "class", "reason"],
)
classification

,column,class,reason
0,patient_id,identifier,"unique key, joins to the EHR"
1,full_name,direct_identifier,names a person
2,age,quasi_identifier,narrows a population when combined
3,postcode,quasi_identifier,narrows a population when combined
4,diagnosis,sensitive,health status
5,readmitted,label,the outcome under study


**Expected output** — six classified columns. The distinction that matters downstream is
`direct_identifier` (must never reach a model) versus `quasi_identifier` (may, but only in combination
controlled by a k-anonymity rule).

### Step 3 — One loader, deny by default

Every read goes through this function. It raises rather than warns, because a warning in a data pipeline
is a message nobody reads at 3 a.m.

In [4]:
ALLOWED = {"label", "quasi_identifier"}   # what a training job may see


class DataPolicyError(RuntimeError):
    pass


def load_for_training(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    unknown = set(columns) - set(df.columns)
    if unknown:
        raise DataPolicyError(f"unknown columns requested: {sorted(unknown)}")
    classes = dict(zip(classification.column, classification["class"]))
    refused = [c for c in columns if classes[c] not in ALLOWED]
    if refused:
        raise DataPolicyError(
            "columns outside the training allowlist: "
            + ", ".join(f"{c} ({classes[c]})" for c in refused)
        )
    return df[columns].copy()


ok = load_for_training(data, ["age", "postcode", "readmitted"])
print("allowed read:", ok.shape)
for bad in (["full_name", "readmitted"], ["diagnosis", "readmitted"], ["dob"]):
    try:
        load_for_training(data, bad)
    except DataPolicyError as exc:
        print("refused:", exc)

allowed read: (8, 3)
refused: columns outside the training allowlist: full_name (direct_identifier)
refused: columns outside the training allowlist: diagnosis (sensitive)
refused: unknown columns requested: ['dob']


**Expected output** — one accepted read of `(8, 3)` and three refusals with reasons. The refusal text is
the deliverable: when a colleague asks why their notebook broke, the message names the column and its
class.

### Step 4 — Minimisation, measured

Minimisation is not a slogan; you can count it. Report how many columns and how many sensitive cells are
exposed by each proposed read.

In [5]:
proposals = {
    "everything": list(data.columns),
    "as requested by a downstream analyst": ["patient_id", "age", "postcode", "diagnosis", "readmitted"],
    "minimum for the model": ["age", "postcode", "readmitted"],
}
rows = []
for name, cols in proposals.items():
    exposed = data[cols]
    rows.append({"proposal": name, "columns": len(cols),
                 "identifier_cells": int(exposed[[c for c in cols if c in
                                                  ("patient_id", "full_name")]].size),
                 "sensitive_cells": int(exposed[[c for c in cols if c == "diagnosis"]].size)})
exposure = pd.DataFrame(rows)
exposure

,proposal,columns,identifier_cells,sensitive_cells
0,everything,6,16,8
1,as requested by a downstream analyst,5,8,8
2,minimum for the model,3,0,0


**Expected output** — the "everything" proposal exposes 16 identifier cells and 8 sensitive cells; the
minimum proposal exposes none. The middle row is the realistic failure: a plausible request that carries
both.

### Step 5 — Integrity, then a dataset card

A dataset card records what a reviewer needs in order to say no: what is in it, what was excluded, how it
was verified, and what it must not be used for.

In [6]:
csv_path = workdir / "training_view.csv"
ok.to_csv(csv_path, index=False)
digest = hashlib.sha256(csv_path.read_bytes()).hexdigest()

card = f'''# Dataset card: triage readmission view
- Rows: {len(ok)} (synthetic)
- Columns: {", ".join(ok.columns)}
- Excluded by policy: {", ".join(sorted(set(data.columns) - set(ok.columns)))}
- SHA-256: {digest[:16]}...
- Intended use: teaching, robustness experiments
- Not for: clinical decisions, re-identification studies, linkage to other datasets
- Known limitation: 8 synthetic rows; no claim of representativeness
'''
print(card)
assert "full_name" not in ok.columns and "diagnosis" not in ok.columns

# Dataset card: triage readmission view
- Rows: 8 (synthetic)
- Columns: age, postcode, readmitted
- Excluded by policy: diagnosis, full_name, patient_id
- SHA-256: 6f16ba41a0f8f186...
- Intended use: teaching, robustness experiments
- Not for: clinical decisions, re-identification studies, linkage to other datasets
- Known limitation: 8 synthetic rows; no claim of representativeness



**Expected output** — a short markdown card with the digest prefix, the exclusion list and the "not for"
section. The assertion is the part that would fail in CI if someone widened the allowlist.

## Exercises

1. **Add a column.** Introduce `phone_number`, classify it, and confirm the loader refuses it without any
   change to the loader itself.
2. **Add a purpose.** Extend the policy so that `diagnosis` is readable by a role `researcher` but not by
   `service`. Keep the default refusal for everyone else.
3. **Break the card.** Remove the SHA-256 line and argue why the card is now less useful to an auditor.
4. **Measure the cost of policy.** Time `load_for_training` on a 1,000,000-row frame and decide whether
   the copy is affordable in your pipeline.

## Reflection questions

1. Why is a column allowlist stronger than training people not to read identifier columns?
2. The card says "not for clinical decisions". Who enforces that, and what would enforcement look like?
3. Module 09 shows that a model can leak its training data. How does minimisation change that risk, and
   what does it leave untouched?

## References

1. NIST, *SP 800-188*, De-Identifying Government Datasets.
2. Information Commissioner's Office, *Data minimisation* principle (UK GDPR Article 5).
3. Gebru, T. et al., "Datasheets for Datasets", 2018.
4. NIST, *AI RMF 1.0*, NIST AI 100-1, 2023 — Map 1.1, data provenance.